# Anomaly Detection Pipeline v5 — Single Optimized Submission

## Diagnosis from v4 Codabench:
- AUC=0.967, P=0.802, R=0.584, F1=0.676
- **Test anomaly rate is ~11-15%** (estimated from v4 P/R — much higher than training 8.5%!)
- Codabench likely thresholds at 0.5 on submitted scores

## v5 Two-Pronged Attack:
### 1. AUC boost via semi-supervised features (KNN-label, target-encoded items, Mahalanobis)
### 2. Rank-sigmoid score transformation — maps cross-batch-calibrated percentile to 0.5


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
os.chdir('/content/drive/My Drive/cs421 principles of machine learning/group project/leong/iter 8')
print('Working directory:', os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import warnings, json, itertools
warnings.filterwarnings("ignore")

from scipy import stats
from scipy.sparse import csr_matrix
from scipy.stats import rankdata
from scipy.spatial.distance import mahalanobis
from sklearn.decomposition import TruncatedSVD, NMF
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, classification_report)
from sklearn.ensemble import (IsolationForest, RandomForestClassifier,
                              GradientBoostingClassifier, ExtraTreesClassifier)
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb
import xgboost as xgb
print("All imports successful.")

In [ ]:
train_data_1 = np.load("training_batch_with_labels.npz")
train_data_2 = np.load("first_batch_with_labels.npz")
train_data_3 = np.load("second_batch_with_labels.npz")
test_data    = np.load("third_batch.npz")

batches = [
    {"X": pd.DataFrame(d["X"], columns=["user","item","rating"]),
     "y": pd.DataFrame(d["y"], columns=["user","label"]).set_index("user")}
    for d in [train_data_1, train_data_2, train_data_3]
]

train_ratings = pd.concat([b["X"] for b in batches], ignore_index=True)
labels = pd.concat([b["y"] for b in batches])
test_ratings = pd.DataFrame(test_data["X"], columns=["user", "item", "rating"])
all_ratings = pd.concat([train_ratings, test_ratings], ignore_index=True)
n_items = 1000

print(f"Training: {train_ratings['user'].nunique()} users, {len(train_ratings)} interactions")
print(f"Test:     {test_ratings['user'].nunique()} users, {len(test_ratings)} interactions")
for i, b in enumerate(batches):
    na = int(b['y']['label'].sum())
    print(f"  Batch {i+1}: {len(b['y'])} users, {na} anomalous ({na/len(b['y'])*100:.1f}%)")
print(f"Overall training anomaly rate: {labels['label'].mean():.1%}")

In [ ]:
g_item_stats = all_ratings.groupby("item")["rating"].agg(["mean", "std", "count"])
g_item_stats.columns = ["item_mean", "item_std", "item_count"]
g_item_stats["item_std"] = g_item_stats["item_std"].fillna(0)
g_item_pop = all_ratings.groupby("item")["rating"].count()
g_mean = all_ratings["rating"].mean()

train_with_labels = train_ratings.merge(labels.reset_index(), on="user", how="left")
item_anomaly_count = train_with_labels.groupby("item")["label"].sum()
item_total_count = train_with_labels.groupby("item").size()
global_anom_rate = labels["label"].mean()
smoothing = 20
item_anomaly_smoothed = (item_anomaly_count + smoothing * global_anom_rate) / (item_total_count + smoothing)

print(f"Global mean rating: {g_mean:.3f}")
print(f"Item anomaly rate range: {item_anomaly_smoothed.min():.3f} - {item_anomaly_smoothed.max():.3f}")

In [ ]:
def compute_base_features(df, svd_model=None, nmf_model=None, svd10_model=None, fit=False):
    feats = []; users_list = sorted(df["user"].unique())

    basic = df.groupby("user")["rating"].agg(
        mean_r="mean", std_r="std", min_r="min", max_r="max",
        med_r="median", n_ratings="count", sum_r="sum").fillna(0)
    basic["range_r"] = basic["max_r"] - basic["min_r"]
    feats.append(basic)

    def dist_f(g):
        r = g["rating"].values; n = len(r); d = {}
        d["skew"] = stats.skew(r) if n >= 3 else 0
        d["kurt"] = stats.kurtosis(r) if n >= 4 else 0
        vc = pd.Series(r).value_counts(normalize=True)
        d["entropy"] = stats.entropy(vc); d["mode_prop"] = vc.iloc[0]
        d["mad"] = np.median(np.abs(r - np.median(r)))
        d["iqr"] = np.percentile(r, 75) - np.percentile(r, 25) if n >= 4 else 0
        d["cv"] = (np.std(r) / np.mean(r)) if np.mean(r) > 0 else 0
        for rv in range(6): d[f"pr{rv}"] = np.mean(r == rv)
        d["p_extreme"] = np.mean((r == 0) | (r == 5))
        d["p_low"] = np.mean(r <= 1); d["p_high"] = np.mean(r >= 4)
        d["n_distinct_r"] = len(np.unique(r))
        d["change_rate"] = np.sum(np.diff(r) != 0) / (n - 1) if n > 1 else 0
        d["concentration"] = np.sum(vc.values ** 2)
        if n > 1:
            diffs = np.diff(r)
            d["mean_abs_diff"] = np.mean(np.abs(diffs)); d["std_diff"] = np.std(diffs)
            d["p_same_rating"] = np.mean(diffs == 0)
            d["max_run"] = max(len(list(gi)) for _, gi in itertools.groupby(r))
        else: d["mean_abs_diff"] = 0; d["std_diff"] = 0; d["p_same_rating"] = 0; d["max_run"] = n
        if n >= 10:
            d["p10"] = np.percentile(r, 10); d["p90"] = np.percentile(r, 90)
            d["p90_p10"] = d["p90"] - d["p10"]
        else: d["p10"] = 0; d["p90"] = 0; d["p90_p10"] = 0
        return pd.Series(d)
    feats.append(df.groupby("user").apply(dist_f))

    inter = df.groupby("user").agg(n_unique_items=("item", "nunique"))
    inter["repeat_ratio"] = 1 - inter["n_unique_items"] / df.groupby("user").size()
    inter["density"] = df.groupby("user").size() / n_items
    ic = df.groupby(["user", "item"]).size().reset_index(name="cnt")
    rs2 = ic.groupby("user")["cnt"].agg(max_item_repeat="max", mean_item_repeat="mean").fillna(0)
    inter = inter.join(rs2, how="left").fillna(0)
    feats.append(inter)

    m = df.merge(g_item_stats, left_on="item", right_index=True, how="left")
    m["item_mean"] = m["item_mean"].fillna(g_mean); m["item_std"] = m["item_std"].fillna(1)
    m["resid"] = m["rating"] - m["item_mean"]
    m["z_resid"] = np.where(m["item_std"] > 0, m["resid"] / m["item_std"], 0)
    m["abs_resid"] = np.abs(m["resid"])
    r_agg = m.groupby("user").agg(
        mean_resid=("resid", "mean"), std_resid=("resid", "std"),
        abs_mean_resid=("abs_resid", "mean"), max_abs_resid=("abs_resid", "max"),
        med_resid=("resid", "median"),
        mean_z=("z_resid", "mean"), std_z=("z_resid", "std"),
        abs_mean_z=("z_resid", lambda x: np.abs(x).mean()),
        resid_q90=("abs_resid", lambda x: np.percentile(x, 90)),
        n_large_dev=("abs_resid", lambda x: (x > 2).sum()),
    ).fillna(0)
    r_agg["large_dev_ratio"] = r_agg["n_large_dev"] / df.groupby("user").size()
    np_ = m.groupby("user")["resid"].apply(lambda x: (x > 1).sum())
    nn_ = m.groupby("user")["resid"].apply(lambda x: (x < -1).sum())
    r_agg["dev_asymmetry"] = (np_ - nn_) / (np_ + nn_ + 1)
    feats.append(r_agg)

    mp = df.merge(g_item_pop.rename("ipop").to_frame(), left_on="item", right_index=True, how="left")
    mp["ipop"] = mp["ipop"].fillna(1); mp["iw"] = 1.0 / (mp["ipop"] + 1); mp["wr"] = mp["rating"] * mp["iw"]
    pf = mp.groupby("user").agg(wr_sum=("wr", "sum"), iw_sum=("iw", "sum"),
        mean_ipop=("ipop", "mean"), std_ipop=("ipop", "std"),
        mean_lp=("ipop", lambda x: np.log1p(x).mean()))
    pf["w_mean_r"] = pf["wr_sum"] / pf["iw_sum"]
    pf = pf.drop(columns=["wr_sum", "iw_sum"]).fillna(0)
    mp["pop_q"] = pd.qcut(mp["ipop"].rank(method="first"), q=4, labels=False)
    for q in range(4): pf[f"mean_r_popq{q}"] = mp[mp["pop_q"] == q].groupby("user")["rating"].mean()
    pf = pf.fillna(0)
    feats.append(pf)

    def seq_f(g):
        r = g["rating"].values; n = len(r); d = {}
        if n >= 3:
            mid = n // 2; d["drift"] = np.mean(r[mid:]) - np.mean(r[:mid])
            d["trend"] = np.polyfit(range(n), r, 1)[0]
            d["autocorr"] = np.corrcoef(r[:-1], r[1:])[0, 1] if np.std(r) > 0 else 0
            if np.isnan(d["autocorr"]): d["autocorr"] = 0
        else: d["drift"] = 0; d["trend"] = 0; d["autocorr"] = 0
        if n >= 10:
            q1, q2, q3, q4 = np.array_split(r, 4)
            d["q4_q1_diff"] = np.mean(q4) - np.mean(q1)
            d["segment_std"] = np.std([np.mean(q1), np.mean(q2), np.mean(q3), np.mean(q4)])
        else: d["q4_q1_diff"] = 0; d["segment_std"] = 0
        return pd.Series(d)
    feats.append(df.groupby("user").apply(seq_f))

    def div_f(g):
        vc = pd.Series(g["item"].values).value_counts(normalize=True)
        return pd.Series({"item_entropy": stats.entropy(vc), "gini_simpson": 1 - np.sum(vc**2),
                          "unique_ratio": len(np.unique(g["item"].values)) / n_items})
    feats.append(df.groupby("user").apply(div_f))

    uid_map = {u: i for i, u in enumerate(users_list)}
    rows = df["user"].map(uid_map).values; cols = df["item"].values
    vals = df["rating"].values.astype(float)
    um = csr_matrix((vals, (rows, cols)), shape=(len(users_list), n_items))
    um_dense = um.toarray()

    nc_svd = 35
    if fit or svd_model is None:
        svd_model = TruncatedSVD(n_components=nc_svd, random_state=42)
        emb = svd_model.fit_transform(um)
    else: emb = svd_model.transform(um)
    svd_df = pd.DataFrame(emb, index=users_list, columns=[f"svd_{i}" for i in range(nc_svd)])
    svd_df.index.name = "user"
    recon35 = svd_model.inverse_transform(emb)
    svd_df["svd_err_35"] = np.mean((um_dense - recon35) ** 2, axis=1)
    svd_df["svd_norm"] = np.linalg.norm(emb, axis=1)

    if fit or svd10_model is None:
        svd10_model = TruncatedSVD(n_components=10, random_state=42)
        emb10 = svd10_model.fit_transform(um)
    else: emb10 = svd10_model.transform(um)
    recon10 = svd10_model.inverse_transform(emb10)
    svd_df["svd_err_10"] = np.mean((um_dense - recon10) ** 2, axis=1)
    svd_df["svd_err_diff"] = svd_df["svd_err_10"] - svd_df["svd_err_35"]
    feats.append(svd_df)

    nc_nmf = 15
    um_pos = um.copy(); um_pos.data = np.clip(um_pos.data, 0, None)
    if fit or nmf_model is None:
        nmf_model = NMF(n_components=nc_nmf, random_state=42, max_iter=300, init='nndsvda')
        ne = nmf_model.fit_transform(um_pos)
    else: ne = nmf_model.transform(um_pos)
    ndf = pd.DataFrame(ne, index=users_list, columns=[f"nmf_{i}" for i in range(nc_nmf)])
    ndf.index.name = "user"
    nr = nmf_model.inverse_transform(ne)
    ndf["nmf_err"] = np.mean((um_pos.toarray() - nr) ** 2, axis=1)
    ndf["nmf_norm"] = np.linalg.norm(ne, axis=1)
    feats.append(ndf)

    bm = csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(users_list), n_items))
    avg = bm.mean(axis=0).A1
    sims = []
    for i in range(len(users_list)):
        uv = bm[i].toarray().flatten()
        d2 = np.dot(uv, avg); nu = np.linalg.norm(uv); na = np.linalg.norm(avg)
        sims.append(d2 / (nu * na) if nu > 0 and na > 0 else 0)
    feats.append(pd.DataFrame({"cos_sim_avg": sims}, index=users_list).rename_axis("user"))

    mp2 = df.merge(g_item_pop.rename("ipop").to_frame(), left_on="item", right_index=True, how="left")
    mp2["ipop"] = mp2["ipop"].fillna(1)
    def pcorr(g):
        if len(g) < 5: return pd.Series({"rating_pop_corr": 0})
        c = np.corrcoef(g["rating"].values, g["ipop"].values)[0, 1]
        return pd.Series({"rating_pop_corr": c if not np.isnan(c) else 0})
    feats.append(mp2.groupby("user").apply(pcorr))

    def burst_f(g):
        items = g["item"].values; n = len(items)
        if n >= 10:
            h1 = set(items[:n//2]); h2 = set(items[n//2:])
            return pd.Series({"item_overlap_halves": len(h1 & h2) / max(len(h1 | h2), 1)})
        return pd.Series({"item_overlap_halves": 0})
    feats.append(df.groupby("user").apply(burst_f))

    m_te = df.merge(item_anomaly_smoothed.rename("item_anom_rate").to_frame(),
                    left_on="item", right_index=True, how="left")
    m_te["item_anom_rate"] = m_te["item_anom_rate"].fillna(global_anom_rate)
    te_agg = m_te.groupby("user")["item_anom_rate"].agg(
        mean_item_anom="mean", std_item_anom="std", max_item_anom="max", min_item_anom="min").fillna(0)
    te_agg["range_item_anom"] = te_agg["max_item_anom"] - te_agg["min_item_anom"]
    m_te2 = m_te.merge(g_item_stats[["item_mean"]], left_on="item", right_index=True, how="left")
    m_te2["item_mean"] = m_te2["item_mean"].fillna(g_mean)
    m_te2["aw_resid"] = (m_te2["rating"] - m_te2["item_mean"]) * m_te2["item_anom_rate"]
    te_agg["anom_w_resid_mean"] = m_te2.groupby("user")["aw_resid"].mean()
    te_agg["anom_w_resid_std"] = m_te2.groupby("user")["aw_resid"].std()
    te_agg = te_agg.fillna(0)
    feats.append(te_agg)

    result = feats[0]
    for f in feats[1:]: result = result.join(f, how="left")
    result = result.fillna(0).replace([np.inf, -np.inf], 0)
    return result, svd_model, nmf_model, svd10_model


def add_semisup_features(train_f, test_f, y_arr):
    sc = RobustScaler()
    tr_sc = sc.fit_transform(train_f); te_sc = sc.transform(test_f)
    for k in [5, 15, 30]:
        nn = NearestNeighbors(n_neighbors=k+1, metric='euclidean', n_jobs=-1)
        nn.fit(tr_sc)
        _, idx_tr = nn.kneighbors(tr_sc)
        train_f[f"knn_anom_k{k}"] = [np.mean(y_arr[[j for j in idx_tr[i] if j != i][:k]]) for i in range(len(tr_sc))]
        _, idx_te = nn.kneighbors(te_sc)
        test_f[f"knn_anom_k{k}"] = [np.mean(y_arr[idx_te[i][:k]]) for i in range(len(te_sc))]
        dists_tr, _ = nn.kneighbors(tr_sc)
        train_f[f"knn_dist_k{k}"] = [np.mean(dists_tr[i, 1:k+1]) for i in range(len(tr_sc))]
        dists_te, _ = nn.kneighbors(te_sc)
        test_f[f"knn_dist_k{k}"] = [np.mean(dists_te[i, :k]) for i in range(len(te_sc))]
    anom_c = tr_sc[y_arr == 1].mean(axis=0); norm_c = tr_sc[y_arr == 0].mean(axis=0)
    for data, lbl in [(tr_sc, train_f), (te_sc, test_f)]:
        lbl["dist_anom_center"] = np.linalg.norm(data - anom_c, axis=1)
        lbl["dist_norm_center"] = np.linalg.norm(data - norm_c, axis=1)
        lbl["dist_ratio"] = lbl["dist_anom_center"] / (lbl["dist_norm_center"] + 1e-8)
    svd_cols = [c for c in train_f.columns if c.startswith("svd_") and c[4:].isdigit()]
    if len(svd_cols) > 0:
        svd_tr = train_f[svd_cols].values; svd_te = test_f[svd_cols].values
        norm_svd = svd_tr[y_arr == 0]
        cov_n = np.cov(norm_svd, rowvar=False) + 1e-6 * np.eye(len(svd_cols))
        cov_inv = np.linalg.inv(cov_n); mean_n = norm_svd.mean(axis=0)
        train_f["mahalanobis"] = [mahalanobis(svd_tr[i], mean_n, cov_inv) for i in range(len(svd_tr))]
        test_f["mahalanobis"] = [mahalanobis(svd_te[i], mean_n, cov_inv) for i in range(len(svd_te))]
    return train_f, test_f


def add_unsup_features(train_f, test_f):
    sc = RobustScaler()
    tr_s = sc.fit_transform(train_f); te_s = sc.transform(test_f)
    for i, (cont, mf, rs) in enumerate([(0.085, 0.8, 42), (0.095, 0.6, 77), (0.09, 0.7, 123)]):
        iso = IsolationForest(n_estimators=300, contamination=cont, random_state=rs, max_features=mf)
        iso.fit(tr_s)
        train_f[f"iso_{i}"] = -iso.score_samples(tr_s); test_f[f"iso_{i}"] = -iso.score_samples(te_s)
    for kl in [15, 25, 40]:
        lof = LocalOutlierFactor(n_neighbors=kl, novelty=True, contamination=0.09)
        lof.fit(tr_s)
        train_f[f"lof_k{kl}"] = -lof.score_samples(tr_s); test_f[f"lof_k{kl}"] = -lof.score_samples(te_s)
    return train_f, test_f


print("Computing base features...")
X_train_f, svd_m, nmf_m, svd10_m = compute_base_features(train_ratings, fit=True)
X_test_f, _, _, _ = compute_base_features(test_ratings, svd_model=svd_m, nmf_model=nmf_m, svd10_model=svd10_m, fit=False)
y_train = labels.loc[X_train_f.index, "label"]
print(f"Base features: {X_train_f.shape[1]}")

print("Adding semi-supervised features...")
X_train_f, X_test_f = add_semisup_features(X_train_f, X_test_f, y_train.values)
print(f"After semi-sup: {X_train_f.shape[1]}")

print("Adding unsupervised features...")
X_train_f, X_test_f = add_unsup_features(X_train_f, X_test_f)
print(f"Final features: {X_train_f.shape[1]}")

## 3. Cross-Batch Percentile Calibration

In [ ]:
print("Cross-Batch Percentile Calibration")
print("=" * 60)

cb_optimal_pctiles = []; cb_f1s = []

for hold_idx in range(3):
    train_b = [b for i, b in enumerate(batches) if i != hold_idx]
    val_b = batches[hold_idx]
    cb_train_r = pd.concat([b["X"] for b in train_b], ignore_index=True)
    cb_labels = pd.concat([b["y"] for b in train_b])
    cb_val_r = val_b["X"]; cb_val_y = val_b["y"]

    cb_tr_f, cb_svd, cb_nmf, cb_s10 = compute_base_features(cb_train_r, fit=True)
    cb_va_f, _, _, _ = compute_base_features(cb_val_r, svd_model=cb_svd, nmf_model=cb_nmf, svd10_model=cb_s10, fit=False)
    cb_y_tr = cb_labels.loc[cb_tr_f.index, "label"].values
    cb_y_val = cb_val_y.loc[cb_va_f.index, "label"].values

    cb_tr_f, cb_va_f = add_semisup_features(cb_tr_f, cb_va_f, cb_y_tr)
    cb_tr_f, cb_va_f = add_unsup_features(cb_tr_f, cb_va_f)

    common_cols = sorted(set(cb_tr_f.columns) & set(cb_va_f.columns))
    cb_X = cb_tr_f[common_cols].values; cb_Xv = cb_va_f[common_cols].values
    cb_spw = (len(cb_y_tr) - cb_y_tr.sum()) / cb_y_tr.sum()

    cb_preds = np.zeros(len(cb_y_val)); n_m = 0
    for rs, lr, md, nl in [(42,0.02,5,20), (77,0.03,4,12), (123,0.025,6,25), (200,0.04,3,8)]:
        m = lgb.LGBMClassifier(n_estimators=500, learning_rate=lr, max_depth=md, num_leaves=nl,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.6,
            reg_alpha=1.0, reg_lambda=2.0, scale_pos_weight=cb_spw, random_state=rs, verbose=-1)
        m.fit(cb_X, cb_y_tr); cb_preds += m.predict_proba(cb_Xv)[:, 1]; n_m += 1
    for rs, lr, md in [(42,0.02,5), (77,0.03,4)]:
        m = xgb.XGBClassifier(n_estimators=500, learning_rate=lr, max_depth=md,
            min_child_weight=5, subsample=0.8, colsample_bytree=0.6,
            reg_alpha=1.0, reg_lambda=2.0, scale_pos_weight=cb_spw,
            random_state=rs, eval_metric="logloss", verbosity=0)
        m.fit(cb_X, cb_y_tr); cb_preds += m.predict_proba(cb_Xv)[:, 1]; n_m += 1
    cb_preds /= n_m

    best_f1_val, best_pctile = 0, 10
    for pct in np.arange(5, 25, 0.5):
        thr = np.percentile(cb_preds, 100 - pct)
        f1 = f1_score(cb_y_val, (cb_preds >= thr).astype(int))
        if f1 > best_f1_val: best_f1_val, best_pctile = f1, pct
    cb_optimal_pctiles.append(best_pctile); cb_f1s.append(best_f1_val)
    print(f"  Batch {hold_idx+1}: opt_pctile={best_pctile:.1f}%, F1={best_f1_val:.3f}, true_rate={cb_y_val.mean()*100:.1f}%")

cb_pctile_final = np.median(cb_optimal_pctiles)
print(f"\nCross-batch optimal percentile: {cb_pctile_final:.1f}%")
print(f"Mean cross-batch F1: {np.mean(cb_f1s):.3f}")

## 4. Model Training

In [ ]:
fcols = X_train_f.columns.tolist()
X = X_train_f[fcols].values; Xt = X_test_f[fcols].values; y = y_train.values
sc2 = RobustScaler(); Xs = sc2.fit_transform(X); Xts2 = sc2.transform(Xt)
spw = (len(y) - y.sum()) / y.sum()
print(f"Imbalance: {spw:.0f}:1, Features: {X.shape[1]}")

def get_models(seed=0):
    return {
        f"lgbm_a_{seed}": lgb.LGBMClassifier(n_estimators=600, learning_rate=0.02, max_depth=5, num_leaves=20,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.6, reg_alpha=0.8, reg_lambda=1.5,
            scale_pos_weight=spw, random_state=42+seed, verbose=-1),
        f"lgbm_b_{seed}": lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03, max_depth=4, num_leaves=12,
            min_child_samples=15, subsample=0.75, colsample_bytree=0.5, reg_alpha=1.5, reg_lambda=2.5,
            scale_pos_weight=spw, random_state=77+seed, verbose=-1),
        f"lgbm_c_{seed}": lgb.LGBMClassifier(n_estimators=550, learning_rate=0.025, max_depth=6, num_leaves=25,
            min_child_samples=8, subsample=0.85, colsample_bytree=0.65, reg_alpha=0.5, reg_lambda=1.0,
            scale_pos_weight=spw, random_state=123+seed, verbose=-1),
        f"lgbm_d_{seed}": lgb.LGBMClassifier(n_estimators=400, learning_rate=0.04, max_depth=3, num_leaves=8,
            min_child_samples=20, subsample=0.7, colsample_bytree=0.45, reg_alpha=2.0, reg_lambda=3.0,
            scale_pos_weight=spw, random_state=200+seed, verbose=-1),
        f"lgbm_dart_{seed}": lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, max_depth=5, num_leaves=16,
            min_child_samples=12, subsample=0.8, colsample_bytree=0.6, reg_alpha=1.0, reg_lambda=2.0,
            boosting_type='dart', is_unbalance=True, random_state=42+seed, verbose=-1),
        f"xgb_a_{seed}": xgb.XGBClassifier(n_estimators=600, learning_rate=0.02, max_depth=5,
            min_child_weight=5, subsample=0.8, colsample_bytree=0.6, reg_alpha=0.8, reg_lambda=1.5,
            scale_pos_weight=spw, random_state=42+seed, eval_metric="logloss", verbosity=0),
        f"xgb_b_{seed}": xgb.XGBClassifier(n_estimators=500, learning_rate=0.03, max_depth=4,
            min_child_weight=8, subsample=0.75, colsample_bytree=0.5, reg_alpha=1.5, reg_lambda=2.5,
            scale_pos_weight=spw, random_state=77+seed, eval_metric="logloss", verbosity=0),
        f"xgb_c_{seed}": xgb.XGBClassifier(n_estimators=550, learning_rate=0.025, max_depth=6,
            min_child_weight=3, subsample=0.85, colsample_bytree=0.65, reg_alpha=0.5, reg_lambda=1.0,
            scale_pos_weight=spw, random_state=123+seed, eval_metric="logloss", verbosity=0),
        f"rf_{seed}": RandomForestClassifier(n_estimators=600, max_depth=10, min_samples_leaf=3,
            class_weight="balanced", random_state=42+seed, n_jobs=-1),
        f"et_{seed}": ExtraTreesClassifier(n_estimators=600, max_depth=10, min_samples_leaf=3,
            class_weight="balanced", random_state=42+seed, n_jobs=-1),
        f"gb_{seed}": GradientBoostingClassifier(n_estimators=400, learning_rate=0.03, max_depth=4,
            min_samples_leaf=8, subsample=0.8, random_state=42+seed),
        f"lr_{seed}": LogisticRegression(C=0.5, class_weight="balanced", max_iter=3000, random_state=42+seed),
    }
print(f"Models per seed: {len(get_models(0))}")

In [ ]:
N_FOLDS = 10
all_oof = {}; all_test_p = {}

for seed in [0, 1000]:
    models = get_models(seed); mnames = list(models.keys())
    oof = {n: np.zeros(len(y)) for n in mnames}
    tp = {n: np.zeros(len(Xt)) for n in mnames}
    print(f"\nSeed={seed}: {N_FOLDS}-fold CV, {len(mnames)} models")
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42+seed)
    for fi, (tidx, vidx) in enumerate(skf.split(X, y)):
        Xtr, Xv, Xtr_s, Xv_s = X[tidx], X[vidx], Xs[tidx], Xs[vidx]
        ytr, yv = y[tidx], y[vidx]
        fold_m = get_models(seed)
        fi_info = []
        for nm in mnames:
            model = fold_m[nm]
            if "lr" in nm:
                model.fit(Xtr_s, ytr); oof[nm][vidx] = model.predict_proba(Xv_s)[:, 1]
                tp[nm] += model.predict_proba(Xts2)[:, 1] / N_FOLDS
            else:
                model.fit(Xtr, ytr); oof[nm][vidx] = model.predict_proba(Xv)[:, 1]
                tp[nm] += model.predict_proba(Xt)[:, 1] / N_FOLDS
            bf = max(f1_score(yv, (oof[nm][vidx] >= t).astype(int)) for t in np.arange(0.05, 0.95, 0.01))
            fi_info.append(f"{nm.split('_'+str(seed))[0]}:{bf:.3f}")
        if fi < 2: print(f"  Fold {fi+1}: {', '.join(fi_info[:6])} ...")
    all_oof.update(oof); all_test_p.update(tp)
    print(f"  Seed {seed} summary:")
    for n in mnames:
        auc = roc_auc_score(y, oof[n])
        bf = max(f1_score(y, (oof[n] >= t).astype(int)) for t in np.arange(0.01, 0.99, 0.005))
        print(f"    {n:20s}: AUC={auc:.4f}, F1={bf:.4f}")
print(f"\nTotal models: {len(all_oof)}")

## 5. Ensemble Selection

In [ ]:
def best_f1_thr(scores, y_true):
    bf, bt = 0, 0.5
    for t in np.arange(0.01, 0.99, 0.005):
        f = f1_score(y_true, (scores >= t).astype(int))
        if f > bf: bf, bt = f, t
    return bf, bt

all_names = list(all_oof.keys())
m_aucs = {n: roc_auc_score(y, all_oof[n]) for n in all_names}
sorted_m = sorted(m_aucs, key=m_aucs.get, reverse=True)
good = [n for n in sorted_m if m_aucs[n] > 0.8]
print(f"Good models (AUC>0.8): {len(good)}")

ens = {}
bnames = set()
for n in all_names:
    for sfx in ["_0", "_1000"]:
        if n.endswith(sfx): bnames.add(n[:-len(sfx)])
for bn in sorted(bnames):
    match = [n for n in all_names if any(n == f"{bn}_{s}" for s in [0, 1000])]
    if len(match) >= 2:
        ens[f"ms_{bn}"] = (np.mean([all_oof[n] for n in match], 0), np.mean([all_test_p[n] for n in match], 0))

for top_n in [5, 8, 10, 15, 20, len(good)]:
    top = good[:top_n]
    ens[f"top{top_n}_eq"] = (np.mean([all_oof[n] for n in top], 0), np.mean([all_test_p[n] for n in top], 0))
    ens[f"top{top_n}_rank"] = (
        np.mean([rankdata(all_oof[n])/len(y) for n in top], 0),
        np.mean([rankdata(all_test_p[n])/len(Xt) for n in top], 0))

for top_n in [10, 15, 20]:
    top = good[:top_n]
    s_X = np.column_stack([all_oof[n] for n in top] + [rankdata(all_oof[n])/len(y) for n in top])
    s_Xt = np.column_stack([all_test_p[n] for n in top] + [rankdata(all_test_p[n])/len(Xt) for n in top])
    m_oof = np.zeros(len(y)); m_test = np.zeros(len(Xt))
    skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for ti, vi in skf2.split(s_X, y):
        for C in [0.05, 0.1, 0.5, 1.0]:
            mm = LogisticRegression(C=C, class_weight="balanced", max_iter=3000, random_state=42)
            mm.fit(s_X[ti], y[ti])
            m_oof[vi] += mm.predict_proba(s_X[vi])[:, 1] / 4
            m_test += mm.predict_proba(s_Xt)[:, 1] / 20
    ens[f"stack_{top_n}"] = (m_oof, m_test)

print(f"Ensemble candidates: {len(ens)}")
print(f"{'Ensemble':25s} {'AUC':>6} {'OOF_F1':>7}")
print("-" * 42)
ens_results = []
for name, (o, t) in ens.items():
    auc = roc_auc_score(y, o); f1, thr = best_f1_thr(o, y)
    ens_results.append((name, auc, f1, thr, o, t))
ens_results.sort(key=lambda x: -x[1])
for name, auc, f1, thr, _, _ in ens_results[:20]:
    print(f"  {name:25s} {auc:.4f} {f1:.4f}")

## 6. Score Transformation + Single Submission

In [ ]:
best_ens = ens_results[0]
ens_name, ens_auc, ens_f1, ens_thr, oof_scores, test_scores = best_ens
print(f"Selected: {ens_name} (AUC={ens_auc:.4f}, OOF F1={ens_f1:.4f})")

def rank_sigmoid_transform(scores, cutoff_pctile, steepness=25):
    ranks = rankdata(scores) / len(scores)
    cutoff = 1 - cutoff_pctile / 100
    return 1 / (1 + np.exp(-steepness * (ranks - cutoff)))

print(f"\nCross-batch suggested percentile: {cb_pctile_final:.1f}%")
print(f"\nEvaluating percentiles on OOF (threshold=0.5):")
print(f"{'Pctile':>6} {'F1@0.5':>7} {'P@0.5':>7} {'R@0.5':>7} {'Pred%':>6}")
print("-" * 42)

best_pct_f1, best_pct = 0, cb_pctile_final
for pct in np.arange(7, 22, 0.5):
    oof_t = rank_sigmoid_transform(oof_scores, pct)
    preds = (oof_t >= 0.5).astype(int)
    f1 = f1_score(y, preds); p = precision_score(y, preds); r = recall_score(y, preds)
    if f1 > best_pct_f1: best_pct_f1 = f1; best_pct = pct
    if pct % 1 == 0:
        marker = " <<<" if abs(pct - best_pct) < 0.1 else ""
        print(f"  {pct:5.1f}% {f1:.4f} {p:.4f} {r:.4f} {preds.mean()*100:5.1f}%{marker}")

print(f"\nBest OOF percentile: {best_pct:.1f}% (F1@0.5={best_pct_f1:.4f})")

# Final: blend cross-batch estimate with OOF-optimal
final_pct = 0.6 * cb_pctile_final + 0.4 * best_pct
print(f"Final percentile: {final_pct:.1f}% (60% cross-batch + 40% OOF)")

final_test_scores = rank_sigmoid_transform(test_scores, final_pct, steepness=25)
final_oof_scores = rank_sigmoid_transform(oof_scores, final_pct, steepness=25)
oof_preds = (final_oof_scores >= 0.5).astype(int)

print(f"\n{'='*60}")
print("FINAL OOF PERFORMANCE (threshold=0.5)")
print(f"{'='*60}")
print(f"  AUC:       {roc_auc_score(y, final_oof_scores):.4f}")
print(f"  F1:        {f1_score(y, oof_preds):.4f}")
print(f"  Precision: {precision_score(y, oof_preds):.4f}")
print(f"  Recall:    {recall_score(y, oof_preds):.4f}")
print(f"  Predicted:  {oof_preds.sum()}/{len(y)} ({oof_preds.mean()*100:.1f}%)")
print()
print(classification_report(y, oof_preds, target_names=["Normal", "Anomalous"]))

In [ ]:
test_users = X_test_f.index.values
pred_dict = {int(u): float(s) for u, s in zip(test_users, final_test_scores)}

np.savez("submission.npz", predictions=final_test_scores)
with open("predictions.json", "w") as f:
    json.dump({"predictions": pred_dict}, f)

res_df = pd.DataFrame({
    "user": test_users, "anomaly_score": final_test_scores,
    "predicted_label": (final_test_scores >= 0.5).astype(int)
}).sort_values("anomaly_score", ascending=False)
res_df.to_csv("predictions.csv", index=False)

n_anom = (final_test_scores >= 0.5).sum()
print("=" * 60)
print("SUBMISSION SAVED")
print("=" * 60)
print(f"Ensemble: {ens_name}")
print(f"Percentile: {final_pct:.1f}%")
print(f"Predicted anomalies: {n_anom}/{len(test_users)} ({n_anom/len(test_users)*100:.1f}%)")
print(f"Score stats: mean={final_test_scores.mean():.4f}, std={final_test_scores.std():.4f}")
print(f"\nTop 20 most anomalous:")
print(res_df.head(20)[["user", "anomaly_score"]].to_string(index=False))